# ArgLLM baseline (DF-QuAD, re-implemented prompts) — DeepSeek-V3 on COM2SENSE (500 claims, sample hash 3d1fbba18f)

Depth-one ArgLLM with the option to decline an argument, strengths given as a number between 0 and 1 (unreadable answers set to 0.5), temperature 0.7, top-p 0.95. Results: 0.5 Base 75.2, Est. Base 78.6. Run on Kaggle; the API key is read from Kaggle secrets.

In [2]:
# ===== DF-QuAD (ArgLLM, depth 1) — DeepSeek + COM2SENSE (same 500-claim sample) =====
!pip install -q openai scikit-learn

import os, re, json, time, pickle, hashlib, sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# ---- model ----
MODEL_NAME = "deepseek-chat"
LABEL      = "DeepSeek-V3"
DATASET    = "COM2SENSE"
RUN_NAME   = "dfquad_deepseek_com2sense"
WORKERS    = 4
client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"),
                base_url="https://api.deepseek.com", timeout=180)

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT = WORK_DIR / f"{RUN_NAME}_ckpt.pkl"

# ---- SAME COM2SENSE sample as our method (seed 42) ----
C2S_REPO = "/kaggle/working/Com2Sense"
if not Path(C2S_REPO).exists():
    os.system(f"git clone -q --depth 1 https://github.com/PlusLabNLP/Com2Sense.git {C2S_REPO}")
raw = pd.DataFrame(json.load(open(f"{C2S_REPO}/data/train.json")))
raw = raw[raw["label"].isin(["True", "False"])].copy()
raw["gold"] = (raw["label"] == "True").astype(int)
raw = raw.sort_values("id").reset_index(drop=True)
SEED = 42
true_part  = raw[raw["gold"] == 1].sample(n=250, random_state=SEED)
false_part = raw[raw["gold"] == 0].sample(n=250, random_state=SEED)
sample = pd.concat([true_part, false_part]).sample(frac=1, random_state=SEED).reset_index(drop=True)
claims = sample["sent"].astype(str).str.strip().tolist()
gold   = sample["gold"].astype(int).tolist()
SAMPLE_HASH = hashlib.md5("|".join(sample["id"]).encode()).hexdigest()[:10]
print("claims:", len(claims), "| true:", sum(gold), "| SAMPLE HASH:", SAMPLE_HASH)
assert SAMPLE_HASH == "3d1fbba18f", "Different sample! It must be the same 500 claims as our method."

# ---- DF-QuAD library (Uncertainpy, from the ArgLLM repo) ----
repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
sys.path.insert(0, repo)
try:
    import Uncertainpy.src.uncertainpy.gradual as grad
    agg_f = grad.semantics.modular.ProductAggregation()
    inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)
    HAVE_UNCERTAINPY = True
except Exception as e:
    print("Uncertainpy not importable, using built-in DF-QuAD only:", e)
    HAVE_UNCERTAINPY = False


# ---- ArgLLM prompts (original, NO facts) ----
def llm_raw(prompt, max_tokens=200, retries=8):
    last = None
    for a in range(1, retries + 1):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
                temperature=0.7, top_p=0.95, max_tokens=max_tokens)
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e; msg = str(e)
            if "402" in msg:
                raise RuntimeError("no API credit left")
            busy = any(c in msg for c in ["429", "502", "503", "504"]) or "rate" in msg.lower()
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def am_prompt(claim, kind):            # kind: "supporting" / "attacking"
    verb = "support" if kind == "supporting" else "attack"
    cond = ("there is a non-zero probability that the claim is true" if kind == "supporting"
            else "there is a non-zero probability that the claim is false")
    return (f"Please provide a single short argument {kind} the following claim. "
            f"Construct the argument so it refers to the truthfulness of the claim. "
            f"Only provide an argument if you think there is a valid and convincing {verb} "
            f"for this claim ({cond}), otherwise return: N/A.\n"
            f"Claim: {claim}\nArgument:")


def ue_arg_prompt(claim, arg, kind):
    return (f"Consider the following claim and an argument {kind} it.\n"
            f"Claim: {claim}\nArgument: {arg}\n"
            f"How strong is this argument, on a scale from 0 (very weak) to 1 (very strong)? "
            f"Answer with a single number between 0 and 1 and nothing else.")


def ue_topic_prompt(claim):
    return (f"Claim: {claim}\n"
            f"How likely is this claim to be true, on a scale from 0 (certainly false) to 1 (certainly true)? "
            f"Answer with a single number between 0 and 1 and nothing else.")


def gen_argument(claim, kind):
    t = llm_raw(am_prompt(claim, kind), max_tokens=120)
    t = t.replace("Argument:", "").strip().strip('"')
    return "N/A" if (not t or "N/A" in t[:10].upper()) else t


def get_lik(prompt):
    t = llm_raw(prompt, max_tokens=10)
    m = re.search(r"(?<![\d.])(0(?:\.\d+)?|1(?:\.0+)?|\.\d+)(?![\d.])", t)
    return (float(m.group(1)), True) if m else (0.5, False)   # default 0.5, as in ArgLLM


def dfquad_manual(tau, sup, att):
    vs = sup if sup is not None else 0.0
    va = att if att is not None else 0.0
    if va >= vs:
        return tau - tau * (va - vs)
    return tau + (1 - tau) * (vs - va)


def dfquad_strength(v, est):
    tau = v["topic_base"] if est else 0.5
    sup = None if v["sup"] == "N/A" else float(v["sup_base"])
    att = None if v["att"] == "N/A" else float(v["att_base"])
    s = dfquad_manual(float(tau), sup, att)
    if HAVE_UNCERTAINPY:
        bag = grad.BAG()
        claim = grad.Argument("db0", v["claim"], float(tau)); bag.arguments["db0"] = claim
        if sup is not None: bag.add_support(grad.Argument("S1", "s", sup), claim)
        if att is not None: bag.add_attack(grad.Argument("A1", "a", att), claim)
        grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
        s_lib = float(bag.arguments["db0"].strength)
        if abs(s_lib - s) > 1e-6:
            print(f"mismatch {s:.4f} vs {s_lib:.4f}")
        s = s_lib
    return s


def run_one(i):
    c = claims[i]
    v = {"claim": c, "gold": gold[i], "unreadable": 0}
    v["sup"] = gen_argument(c, "supporting")
    v["att"] = gen_argument(c, "attacking")
    v["sup_base"] = v["att_base"] = None
    if v["sup"] != "N/A":
        v["sup_base"], ok = get_lik(ue_arg_prompt(c, v["sup"], "supporting")); v["unreadable"] += not ok
    if v["att"] != "N/A":
        v["att_base"], ok = get_lik(ue_arg_prompt(c, v["att"], "attacking")); v["unreadable"] += not ok
    v["topic_base"], ok = get_lik(ue_topic_prompt(c)); v["unreadable"] += not ok
    v["s_05"] = dfquad_strength(v, False)
    v["s_est"] = dfquad_strength(v, True)
    return v


def safe_one(i):
    try:
        return i, run_one(i), None
    except Exception as e:
        return i, None, str(e)[:80]


def load_ck():
    return pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}


def save_ck(d):
    tmp = CKPT.with_suffix(".tmp"); pickle.dump(d, open(tmp, "wb")); tmp.replace(CKPT)


def report(res):
    idx = sorted(res); y = [res[i]["gold"] for i in idx]; rows = []
    for name, col in [("DF-QuAD 0.5 Base", "s_05"), ("DF-QuAD Est. Base", "s_est")]:
        p = [int(res[i][col] > 0.5) for i in idx]
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
        rows.append({"method": name, "n": len(idx),
                     "acc": round(accuracy_score(y, p) * 100, 1),
                     "prec": round(precision_score(y, p, zero_division=0) * 100, 1),
                     "rec": round(recall_score(y, p, zero_division=0) * 100, 1),
                     "f1": round(f1_score(y, p, zero_division=0) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return pd.DataFrame(rows)


# ---- run (resumable) ----
for attempt in range(3):
    d = load_ck()
    todo = [i for i in range(len(claims)) if i not in d]
    print(f"[pass {attempt + 1}] have {len(d)}/{len(claims)}, running {len(todo)}", flush=True)
    if not todo:
        break
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for f in as_completed([pool.submit(safe_one, i) for i in todo]):
            i, v, err = f.result()
            if err:
                print(f"  error at {i}: {err}", flush=True); continue
            d[i] = v; save_ck(d)
            if len(d) % 50 == 0:
                print(f"  {len(d)}/{len(claims)} | {(time.time() - t0) / 60:.0f} min", flush=True)
    time.sleep(10)

d = load_ck()
table = report(d)
pd.set_option("display.width", 220)
print(f"\n=== FINAL: {LABEL} | {DATASET} | DF-QuAD (no facts) | sample {SAMPLE_HASH} ({len(d)}/{len(claims)}) ===")
print(table.to_string(index=False))
print("N/A sup:", sum(v["sup"] == "N/A" for v in d.values()),
      "| N/A att:", sum(v["att"] == "N/A" for v in d.values()),
      "| unreadable strength scores (default 0.5):", sum(v["unreadable"] for v in d.values()))
table.to_csv(f"/kaggle/working/{RUN_NAME}_results.csv", index=False)

claims: 500 | true: 250 | SAMPLE HASH: 3d1fbba18f
[pass 1] have 0/500, running 500
  50/500 | 1 min
  100/500 | 2 min
  150/500 | 3 min
  200/500 | 4 min
  250/500 | 5 min
  300/500 | 6 min
  350/500 | 7 min
  400/500 | 8 min
  450/500 | 9 min
  500/500 | 10 min
[pass 2] have 500/500, running 0

=== FINAL: DeepSeek-V3 | COM2SENSE | DF-QuAD (no facts) | sample 3d1fbba18f (500/500) ===
           method   n  acc  prec  rec   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 500 75.2  80.3 66.8 72.9 209  41  83 167
DF-QuAD Est. Base 500 78.6  80.4 75.6 77.9 204  46  61 189
N/A sup: 182 | N/A att: 1 | unreadable strength scores (default 0.5): 0
